In [3]:
import os
path_to_main_experiment_folder =  "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
folders = os.listdir(path_to_main_experiment_folder)
# sort folders by name
folders.sort()
print(folders)

['Mask2Former_rios_rgb_FAPN_1', 'Mask2Former_rios_rgb_FAPN_2', 'Mask2Former_rios_rgb_FAPN_3', 'Mask2Former_rios_rgb_FAPN_4', 'Mask2Former_rios_rgb_FAPN_5', 'Mask2Former_rios_rgb_Standard_2_2', 'Mask2Former_rios_rgb_Standard_2_3', 'Mask2Former_rios_rgb_Standard_2_4', 'Mask2Former_rios_rgb_Standard_2_5', 'Mask2Former_rios_rgb_standard_2_1', 'Mask2former_FPN_RGB_1', 'Mask2former_FPN_RGB_1.log', 'Mask2former_FPN_RGB_2', 'Mask2former_FPN_RGB_3', 'Mask2former_FPN_RGB_4', 'Mask2former_FPN_RGB_5']


In [3]:
import json
import os
import math
 
def sanitize_class_metrics_from_json(path_to_metrics_json):
    """
    Reads a Detectron2 metrics.json file, finds the last evaluation entry,
    and returns a dictionary of {class_name: (accuracy, iou, boundary_iou, min_iou)}.
    """
    if not os.path.exists(path_to_metrics_json):
        print(f"Error: File not found at {path_to_metrics_json}")
        return {}
 
    last_valid_metrics = None
 
    with open(path_to_metrics_json, 'r') as f:
        lines = f.readlines()
        
        for line in reversed(lines):
            line = line.strip()
            if not line:
                continue
            
            try:
                data = json.loads(line)
                if any(k.startswith("sem_seg/ACC-") for k in data.keys()):
                    last_valid_metrics = data
                    break
            except json.JSONDecodeError:
                continue
    
    if last_valid_metrics is None:
        print("No valid semantic segmentation class metrics found in the file.")
        return {}
 
    results = {}
 
    for key, value in last_valid_metrics.items():
        if key.startswith("sem_seg/ACC-"):
            class_name = key.replace("sem_seg/ACC-", "")
            
            acc = value
            
            iou_key = f"sem_seg/IoU-{class_name}"
            iou = last_valid_metrics.get(iou_key, None)
 
            b_iou_key = f"sem_seg/BoundaryIoU-{class_name}"
            b_iou = last_valid_metrics.get(b_iou_key, None)
 
            min_iou_key = f"sem_seg/min(IoU, B-Iou)-{class_name}"
            min_iou = last_valid_metrics.get(min_iou_key, None)
            
            results[class_name] = (acc, iou, b_iou, min_iou)
 
    return results
 
 
def _std(values):
    """Population standard deviation of a list of floats."""
    n = len(values)
    if n == 0:
        return float('nan')
    mean = sum(values) / n
    variance = sum((x - mean) ** 2 for x in values) / n
    return math.sqrt(variance)
 
 
def compute_average_metrics(list_of_paths):
    """
    Returns a dictionary of:
      {class_name: (avg_acc, std_acc, avg_iou, std_iou,
                    avg_b_iou, std_b_iou, avg_min_iou, std_min_iou)}
    """
    metrics = []
    for metrics_paths in list_of_paths:
        path_to_metrics_json = os.path.join(metrics_paths, "metrics.json")
        class_metrics = sanitize_class_metrics_from_json(path_to_metrics_json)
        if class_metrics:
            metrics.append(class_metrics)
    
    if not metrics:
        return {}
 
    average_metrics = {}
    all_classes = metrics[0].keys()
 
    for class_name in all_classes:
        accs, ious, b_ious, min_ious = [], [], [], []
 
        for metric in metrics:
            if class_name in metric:
                acc, iou, b_iou, min_iou = metric[class_name]
 
                if acc is not None and not math.isnan(acc):
                    accs.append(acc)
                if iou is not None and not math.isnan(iou):
                    ious.append(iou)
                if b_iou is not None and not math.isnan(b_iou):
                    b_ious.append(b_iou)
                if min_iou is not None and not math.isnan(min_iou):
                    min_ious.append(min_iou)
 
        avg_acc     = sum(accs)     / len(accs)     if accs     else float('nan')
        avg_iou     = sum(ious)     / len(ious)     if ious     else float('nan')
        avg_b_iou   = sum(b_ious)   / len(b_ious)   if b_ious   else float('nan')
        avg_min_iou = sum(min_ious) / len(min_ious) if min_ious else float('nan')
 
        std_acc     = _std(accs)
        std_iou     = _std(ious)
        std_b_iou   = _std(b_ious)
        std_min_iou = _std(min_ious)
 
        average_metrics[class_name] = (
            avg_acc, std_acc,
            avg_iou, std_iou,
            avg_b_iou, std_b_iou,
            avg_min_iou, std_min_iou,
        )
 
    return average_metrics
 
 
def tabulate_metrics_to_markdown(metrics):
    """
    Converts a dictionary of
      {class_name: (avg_acc, std_acc, avg_iou, std_iou,
                    avg_b_iou, std_b_iou, avg_min_iou, std_min_iou)}
    into a Markdown table string showing mean ± std for every metric.
    """
    if not metrics:
        return "No metrics available to tabulate."
 
    header = (
        "| Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |\n"
        "| :--- | :---: | :---: | :---: | :---: |\n"
    )
 
    def fmt(mean, std):
        """Format a mean ± std pair, or 'NaN' if either is not a number."""
        def is_nan(v):
            return v is None or (isinstance(v, float) and math.isnan(v))
        if is_nan(mean):
            return "NaN"
        if is_nan(std):
            return f"{mean:.2f}"
        return f"{mean:.2f} ± {std:.2f}"
 
    rows = []
    for class_name in sorted(metrics.keys()):
        values = metrics[class_name]
 
        if len(values) == 8:
            avg_acc, std_acc, avg_iou, std_iou, avg_b_iou, std_b_iou, avg_min_iou, std_min_iou = values
        elif len(values) == 4:
            # Legacy 4-tuple without std — display mean only
            avg_acc, avg_iou, avg_b_iou, avg_min_iou = values
            std_acc = std_iou = std_b_iou = std_min_iou = float('nan')
        else:
            avg_acc, avg_iou = values[:2]
            avg_b_iou = avg_min_iou = float('nan')
            std_acc = std_iou = std_b_iou = std_min_iou = float('nan')
 
        rows.append(
            f"| {class_name} "
            f"| {fmt(avg_acc, std_acc)} "
            f"| {fmt(avg_iou, std_iou)} "
            f"| {fmt(avg_b_iou, std_b_iou)} "
            f"| {fmt(avg_min_iou, std_min_iou)} |\n"
        )
 
    return header + "".join(rows)

In [8]:
import os
path_to_main_experiment_folder =  "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
folders = os.listdir(path_to_main_experiment_folder)
# sort folders by name
folders.sort()
print(folders)

path_to_test_json = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios/Mask2Former_rios_rgb_FAPN_1/metrics.json"
metrics = sanitize_class_metrics_from_json(path_to_test_json)
print(metrics)

experiment_path = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas a80 rios"
list_of_experiments = [os.path.join(experiment_path, folder) for folder in folders]
fapn_list = [path for path in list_of_experiments if "FAPN" in path]
Standard_list = [path for path in list_of_experiments if "Standard" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['Mask2Former_rios_rgb_FAPN_1', 'Mask2Former_rios_rgb_FAPN_2', 'Mask2Former_rios_rgb_FAPN_3', 'Mask2Former_rios_rgb_FAPN_4', 'Mask2Former_rios_rgb_FAPN_5', 'Mask2Former_rios_rgb_Standard_2_1', 'Mask2Former_rios_rgb_Standard_2_2', 'Mask2Former_rios_rgb_Standard_2_3', 'Mask2Former_rios_rgb_Standard_2_4', 'Mask2Former_rios_rgb_Standard_2_5', 'Mask2former_FPN_RGB_1', 'Mask2former_FPN_RGB_2', 'Mask2former_FPN_RGB_3', 'Mask2former_FPN_RGB_4', 'Mask2former_FPN_RGB_5']
{'Asphalt': (96.35847410166578, 90.60404739307474, 3.907173045820504, 3.907173045820504), 'Bare soil': (92.31057306755174, 84.07540027301111, 5.231094844308878, 5.231094844308878), 'Concrete': (77.4352285527954, 70.96633975386455, 20.923323219449376, 20.923323219449376), 'Eucalyptus': (96.28301180673355, 89.38051683150977, 59.10558158595448, 59.10558158595448), 'Meadows': (93.54667494011116, 90.44299247206455, 36.83782604854562, 36.83782604854562), 'Native trees': (95.39228937828993, 88.83477738564032, 13.552497669622232, 13.552

In [10]:
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for rios dataset:\n", md_table_fapn)
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for rios dataset:\n", md_table_standard)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for rios dataset:\n", md_table_fpn)

FAPN Average Metrics for rios dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 97.78 | 87.27 | 3.95 | 3.95 |
| Bare soil | 92.35 | 84.17 | 5.82 | 5.82 |
| Concrete | 73.82 | 68.59 | 20.89 | 20.89 |
| Eucalyptus | 95.84 | 88.37 | 58.10 | 58.10 |
| Meadows | 93.15 | 90.06 | 33.42 | 33.42 |
| Native trees | 95.01 | 87.82 | 12.93 | 12.93 |
| Pines | 87.00 | 80.35 | 1.08 | 1.08 |
| Rock | 84.85 | 79.54 | 4.53 | 4.53 |
| Tiles | 99.69 | 92.13 | 0.79 | 0.79 |
| Water | 85.42 | 79.05 | 0.77 | 0.77 |
| unlabeled | NaN | NaN | 69.24 | NaN |

Standard Average Metrics for rios dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 97.45 | 86.08 | 4.01 | 4.01 |
| Bare soil | 92.57 | 84.10 | 5.91 | 5.91 |
| Concrete | 43.22 | 41.12 | 20.19 | 20.19 |
| Eucalyptus | 95.26 | 89.25 | 58.97 | 58.97 |
| Meadows | 94.18 | 90.79 | 36.55 | 36.

In [11]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "fapn" in path]
Standard_list = [path for path in list_of_experiments if "std" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_4', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_3', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_1', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_4', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_5', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_3', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_2', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_FPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_std_1', '/home/pablo.canosa/Datos/pruebas ctcomp/pruebas_LADOS/outputs_LADOS/lados_fapn_2'

In [12]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for LADOS dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for LADOS dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for LADOS dataset:\n", md_table_fpn)

Standard Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | NaN | 86.70 | NaN |
| Emulsion | 94.19 | 87.81 | 32.58 | 32.58 |
| Oil | 95.42 | 89.24 | 38.85 | 38.85 |
| Oil-platform | 77.12 | 71.49 | 5.00 | 5.00 |
| Sheen | 86.05 | 80.02 | 21.37 | 21.37 |
| Ship | 83.21 | 78.69 | 1.84 | 1.84 |

FAPN Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | NaN | 88.03 | NaN |
| Emulsion | 94.79 | 87.84 | 38.76 | 38.76 |
| Oil | 95.21 | 89.49 | 42.80 | 42.80 |
| Oil-platform | 81.41 | 80.61 | 5.05 | 5.05 |
| Sheen | 85.49 | 79.40 | 20.15 | 20.15 |
| Ship | 84.07 | 79.50 | 1.93 | 1.93 |

FPN Average Metrics for LADOS dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Background | NaN | N

# Five Billion Pixels

In [13]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "FAPN" in path]
Standard_list = [path for path in list_of_experiments if "MSD" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)


['/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_4', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_4', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_MSD_2_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FAPN_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_gaofen_rgb/Mask2Former_FPN_5', '/home/pablo.canosa/Datos/pruebas ct

In [14]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for FBP dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for FBP dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for FBP dataset:\n", md_table_fpn)

Standard Average Metrics for FBP dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| airport | 67.85 | 57.20 | 0.00 | 0.00 |
| arbor forest | 93.75 | 91.07 | 22.71 | 22.71 |
| artificial meadow | 59.50 | 45.09 | 0.00 | 0.00 |
| bareland | 86.53 | 78.04 | 0.00 | 0.00 |
| dry cropland | 79.59 | 72.27 | 10.28 | 10.28 |
| fish pond | 86.68 | 71.39 | 0.00 | 0.00 |
| garden land | 53.53 | 41.71 | 2.74 | 2.74 |
| industrial area | 80.27 | 71.48 | 9.78 | 9.78 |
| irrigated field | 96.41 | 90.72 | 36.25 | 36.25 |
| lake | 91.58 | 74.78 | 0.00 | 0.00 |
| natural meadow | 90.69 | 83.35 | 24.08 | 24.08 |
| overpass | 68.91 | 61.41 | 0.00 | 0.00 |
| paddy field | 77.47 | 68.76 | 6.56 | 6.56 |
| park | 32.08 | 30.01 | 6.08 | 6.08 |
| pond | 41.61 | 32.62 | 0.00 | 0.00 |
| railway station | 59.81 | 38.49 | 0.00 | 0.00 |
| river | 69.41 | 61.63 | 0.00 | 0.00 |
| road | 83.14 | 68.30 | 0.00 | 0.00 |
| rural residential | 85.78 | 

## RIOS CON AUGMENTED DATASET

In [5]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "FAPN" in path]
Standard_list = [path for path in list_of_experiments if "MSD" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FPN_RGB_TS025_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_MSD_RGB_TS025_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FAPN_RGB_TS025_4', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FAPN_RGB_TS025_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FPN_RGB_TS025_2', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FAPN_RGB_TS025_1', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FPN_RGB_TS025_5', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FAPN_RGB_TS025_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_MSD_RGB_TS025_3', '/home/pablo.canosa/Datos/pruebas ctcomp/output_threshold025_rios/Mask2former_FPN_RGB_TS025_4', '/home/pablo.canosa/Datos/pruebas c

In [6]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for RIOS AUGMENTED dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for RIOS AUGMENTED dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for RIOS AUGMENTED dataset:\n", md_table_fpn)

Standard Average Metrics for RIOS AUGMENTED dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 95.94 ± 0.89 | 91.23 ± 0.97 | 3.88 ± 0.21 | 3.88 ± 0.21 |
| Bare soil | 91.28 ± 0.50 | 83.98 ± 0.91 | 4.95 ± 0.86 | 4.95 ± 0.86 |
| Concrete | 75.70 ± 13.41 | 72.95 ± 13.00 | 22.08 ± 1.45 | 22.08 ± 1.45 |
| Eucalyptus | 96.19 ± 0.48 | 84.09 ± 2.21 | 56.39 ± 0.93 | 56.39 ± 0.93 |
| Meadows | 90.56 ± 1.30 | 88.10 ± 1.05 | 39.11 ± 2.32 | 39.11 ± 2.32 |
| Native trees | 90.17 ± 4.44 | 78.38 ± 3.83 | 11.87 ± 0.94 | 11.87 ± 0.94 |
| Pines | 91.85 ± 8.17 | 86.43 ± 8.28 | 0.92 ± 0.28 | 0.92 ± 0.28 |
| Rock | 85.91 ± 3.13 | 80.60 ± 3.13 | 4.16 ± 0.17 | 4.16 ± 0.17 |
| Tiles | 99.78 ± 0.14 | 97.37 ± 1.68 | 0.96 ± 0.14 | 0.96 ± 0.14 |
| Water | 85.79 ± 2.68 | 80.39 ± 3.25 | 0.69 ± 0.08 | 0.69 ± 0.08 |
| unlabeled | NaN | NaN | 67.90 ± 0.87 | NaN |

FAPN Average Metrics for RIOS AUGMENTED dataset:
 | Class Name | Accurac

# Rios 256 con augmented dataset

In [6]:
path_to_lados_exp = "/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/"
folders = os.listdir(path_to_lados_exp)
list_of_experiments = [os.path.join(path_to_lados_exp, folder) for folder in folders]
print(list_of_experiments)
fapn_list = [path for path in list_of_experiments if "FaPN" in path]
Standard_list = [path for path in list_of_experiments if "MSD" in path]
FPN_list = [path for path in list_of_experiments if "FPN" in path]
print("FAPN experiments:", fapn_list)
print("Standard experiments:", Standard_list)
print("FPN experiments:", FPN_list)
average_fapn_metrics = compute_average_metrics(fapn_list)
average_standard_metrics = compute_average_metrics(Standard_list)
average_fpn_metrics = compute_average_metrics(FPN_list)

['/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FPN_RGB_TS025_1', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FaPN_RGB_TS025_4', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_MSD_RGB_TS025_5', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FPN_RGB_TS025_2', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FaPN_RGB_TS025_1', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FaPN_RGB_TS025_5', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_MSD_RGB_TS025_3', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_FaPN_RGB_TS025_2', '/home/pablo.canosa/Datos/pruebas ctcomp/rios_256_augmented/outputs_rios_256_lado/Mask2former_MSD_R

In [7]:
md_table_standard = tabulate_metrics_to_markdown(average_standard_metrics)
print("Standard Average Metrics for RIOS AUGMENTED dataset:\n", md_table_standard)
md_table_fapn = tabulate_metrics_to_markdown(average_fapn_metrics)
print("FAPN Average Metrics for RIOS AUGMENTED dataset:\n", md_table_fapn)
md_table_fpn = tabulate_metrics_to_markdown(average_fpn_metrics)
print("FPN Average Metrics for RIOS AUGMENTED dataset:\n", md_table_fpn)

Standard Average Metrics for RIOS AUGMENTED dataset:
 | Class Name | Accuracy (%) | IoU (%) | Boundary IoU (%) | Min IoU (%) |
| :--- | :---: | :---: | :---: | :---: |
| Asphalt | 99.38 ± 0.19 | 97.40 ± 1.66 | 3.44 ± 0.13 | 3.44 ± 0.13 |
| Bare soil | 91.73 ± 0.15 | 87.06 ± 0.28 | 10.55 ± 0.44 | 10.55 ± 0.44 |
| Concrete | 99.78 ± 0.04 | 93.87 ± 2.44 | 16.32 ± 0.49 | 16.32 ± 0.49 |
| Eucalyptus | 97.15 ± 0.27 | 89.49 ± 0.36 | 64.52 ± 0.20 | 64.52 ± 0.20 |
| Meadows | 92.20 ± 0.19 | 88.84 ± 0.15 | 51.21 ± 0.90 | 51.21 ± 0.90 |
| Native trees | 97.45 ± 0.54 | 91.57 ± 0.58 | 17.67 ± 0.66 | 17.67 ± 0.66 |
| Pines | 96.63 ± 1.76 | 94.31 ± 3.78 | 1.59 ± 0.02 | 1.59 ± 0.02 |
| Rock | 93.52 ± 1.71 | 85.89 ± 2.72 | 5.68 ± 0.16 | 5.68 ± 0.16 |
| Tiles | 99.70 ± 0.59 | 99.65 ± 0.65 | 0.65 ± 0.09 | 0.65 ± 0.09 |
| Water | 97.83 ± 0.39 | 95.23 ± 1.12 | 0.85 ± 0.05 | 0.85 ± 0.05 |
| unlabeled | NaN | NaN | 73.54 ± 0.39 | NaN |

FAPN Average Metrics for RIOS AUGMENTED dataset:
 | Class Name | Accurac